In [218]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rng = np.random.default_rng(100)

def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

# Initial test of difference with 1000 individuals to predict n for future attempts

## get about 500 people each and bootstrap 

In [219]:


p_A = 10.00
p_B = 15.00
size = 500
power_n=[]

sales_A = sales(p_A, mean_customers=size, rng=rng) * p_A # Compute revenue for each customer in A
sales_B = sales(p_B, mean_customers=size, rng=rng) * p_B # Compute revenue for each customer in B
print(f'There are {len(sales_A)} customers in A and {len(sales_B)} customers in B')

k=2000
for i in range(k):
    boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
    boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)

    revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
    revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B

    delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer

    #Lehr's power model
    power_estimation=16*(np.var(boot_sales_a, ddof=1)+np.var(boot_sales_b, ddof=1))/(delta_R**2)
    #round up
    power_n.append(int(power_estimation)+1)

print(f"Using the Lehr's model for sample size, to have an 80% power rate, our n should be about {np.mean(power_n)}")

There are 525 customers in A and 474 customers in B
Using the Lehr's model for sample size, to have an 80% power rate, our n should be about 467.015


# Next, we do a grid search of 5-70 dollars in $5 increments with n=500

## draw them, then bootstrap to simulate the experiment many times

In [220]:
grid=np.linspace(start=10, stop=70, num=13)

bestprice=5
secondbestprice=0

for i in range(len(grid)):
    p_A = bestprice
    p_B = grid[i]
    size = 500
    sales_A = sales(p_A, mean_customers=size, rng=rng) * p_A 
    sales_B = sales(p_B, mean_customers=size, rng=rng) * p_B 
    k=2000
    bestprice_boot=[]

    for j in range(k):
        boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
        boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)
    
        revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
        revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B
        delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer
        std_error = np.sqrt(
            np.var(sales_A, ddof=1) / len(sales_A) +
            np.var(sales_B, ddof=1) / len(sales_B)
        ) # Standard Error

        confidence_interval = (delta_R - 1.96 * std_error, delta_R + 1.96 * std_error)

        #determine which price is better
        if confidence_interval[0]<=0 and confidence_interval[1]<=0:
            #print(f'{p_A} is better than {p_B}')
            bestprice_boot.append(p_A)
            continue
        elif confidence_interval[0]>=0 and confidence_interval[1]>=0 == np.True_:
            bestprice_boot.append(p_B)
            #print(f'{p_B} is better than {p_A}')
            
        else:
            #print(f'Could not decide between {p_A} and {p_B}')
            bestprice_boot.append(p_B)
            continue
    bestprice = np.median(bestprice_boot)
    print(f'Between {p_A} and {p_B}, the better price is {bestprice}')
            

Between 5 and 10.0, the better price is 10.0
Between 10.0 and 15.0, the better price is 10.0
Between 10.0 and 20.0, the better price is 10.0
Between 10.0 and 25.0, the better price is 25.0
Between 25.0 and 30.0, the better price is 30.0
Between 30.0 and 35.0, the better price is 30.0
Between 30.0 and 40.0, the better price is 30.0
Between 30.0 and 45.0, the better price is 30.0
Between 30.0 and 50.0, the better price is 30.0
Between 30.0 and 55.0, the better price is 30.0
Between 30.0 and 60.0, the better price is 30.0
Between 30.0 and 65.0, the better price is 30.0
Between 30.0 and 70.0, the better price is 30.0
